In [3]:
from pathlib import Path

import hashlib
import re
import wave

import numpy as np
import pandas as pd

from scipy.io import wavfile
from scipy.signal import resample_poly

In [4]:
# ============================================================
# AI-03 configuration
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset"
)

# Raw datasets
ESC_RAW = DATASET_ROOT / "raw" / "esc50"
COSWARA_RAW = DATASET_ROOT / "raw" / "coswara"
AUDIOSET_RAW = DATASET_ROOT / "raw" / "audioset_snoring"

# AI-02 manifests
ESC_MANIFEST = (
    DATASET_ROOT
    / "metadata"
    / "esc50"
    / "source_manifest.csv"
)

COSWARA_MANIFEST = (
    DATASET_ROOT
    / "metadata"
    / "coswara"
    / "source_manifest.csv"
)

AUDIOSET_MANIFEST = (
    DATASET_ROOT
    / "metadata"
    / "audioset_snoring"
    / "source_manifest.csv"
)

AUDIOSET_EVENTS = (
    DATASET_ROOT
    / "metadata"
    / "audioset_snoring"
    / "events.csv"
)

# Output
OUTPUT_ROOT = (
    DATASET_ROOT
    / "processed"
    / "clips_16k_1s"
)

AI03_META = (
    DATASET_ROOT
    / "metadata"
    / "ai03"
)

CLIP_MANIFEST = AI03_META / "clip_manifest.csv"
GENERATION_LOG = AI03_META / "clip_generation_log.csv"

for path in [
    OUTPUT_ROOT / "OTHER",
    OUTPUT_ROOT / "SNORE",
    OUTPUT_ROOT / "COUGH",
    AI03_META,
]:
    path.mkdir(parents=True, exist_ok=True)


# ============================================================
# Audio parameters
# ============================================================

OUTPUT_SR = 16000
OUTPUT_SAMPLES = 16000

RMS_WINDOW_SEC = 0.200
RMS_WINDOW_SAMPLES = int(
    OUTPUT_SR * RMS_WINDOW_SEC
)

ANCHOR_MIN_SEC = 0.4
ANCHOR_MAX_SEC = 0.6

RANDOM_SEED = 2026

#### WAV読み込み・mono化・resampling

In [5]:
def load_and_resample_wav(path):
    """
    WAVを読み込み、
    mono / 16 kHz / int16に変換する。

    Peak normalizationは行わない。
    """

    sr, x = wavfile.read(path)

    # 今回取得したdatasetはすべてPCM integerを想定
    if not np.issubdtype(x.dtype, np.integer):
        raise ValueError(
            f"Unsupported WAV dtype: {x.dtype}"
        )

    # float64へ移してから演算。
    # PCMの数値スケール自体は維持する。
    x = x.astype(np.float64)

    # stereo/multi-channel -> mono
    if x.ndim == 2:
        x = np.mean(x, axis=1)

    elif x.ndim != 1:
        raise ValueError(
            f"Unsupported audio shape: {x.shape}"
        )

    # Resampling
    if sr != OUTPUT_SR:

        gcd = np.gcd(sr, OUTPUT_SR)

        up = OUTPUT_SR // gcd
        down = sr // gcd

        x = resample_poly(
            x,
            up,
            down,
        )

    # int16へ戻す
    #
    # normalizationはしない。
    # resamplingにより僅かにrange外へ出る可能性だけclipする。
    x = np.clip(
        np.rint(x),
        -32768,
        32767,
    ).astype(np.int16)

    return x

#### RMSの算出

In [22]:
def find_max_rms_anchor(
    x,
    region_start_sec=None,
    region_end_sec=None,
):
    """
    1秒clipを生成可能な範囲の中から、
    200 ms RMSが最大となるwindow中心を返す。

    AudioSetではさらにStrong annotation範囲内に限定する。
    """

    n = len(x)
    w = RMS_WINDOW_SAMPLES

    if n < w:
        return None, None

    duration_sec = n / OUTPUT_SR

    # --------------------------------------------------------
    # anchorとして許されるsource上の範囲
    #
    # clip_anchorを0.4～0.6秒に置きながら
    # 1秒clipを完全に収めるために必要。
    # --------------------------------------------------------

    center_min_sec = ANCHOR_MIN_SEC
    center_max_sec = duration_sec - ANCHOR_MIN_SEC

    # AudioSet Strong annotationがある場合は
    # event範囲とのintersectionを取る
    if region_start_sec is not None:
        center_min_sec = max(
            center_min_sec,
            float(region_start_sec),
        )

    if region_end_sec is not None:
        center_max_sec = min(
            center_max_sec,
            float(region_end_sec),
        )

    if center_min_sec > center_max_sec:
        return None, None

    # --------------------------------------------------------
    # RMS
    # --------------------------------------------------------

    x64 = x.astype(np.float64)
    sq = x64 * x64

    cumsum = np.concatenate([
        np.array([0.0]),
        np.cumsum(sq),
    ])

    window_sum = (
        cumsum[w:]
        - cumsum[:-w]
    )

    half_w = w / 2

    # window中心位置からwindow開始sampleへ変換
    start_min = int(np.ceil(
        center_min_sec * OUTPUT_SR
        - half_w
    ))

    start_max = int(np.floor(
        center_max_sec * OUTPUT_SR
        - half_w
    ))

    max_window_start = n - w

    start_min = max(
        0,
        start_min,
    )

    start_max = min(
        max_window_start,
        start_max,
    )

    if start_min > start_max:
        return None, None

    local = window_sum[
        start_min:start_max + 1
    ]

    if len(local) == 0:
        return None, None

    best_start = (
        start_min
        + int(np.argmax(local))
    )

    rms = np.sqrt(
        window_sum[best_start] / w
    )

    anchor_sample = (
        best_start
        + half_w
    )

    anchor_sec = (
        anchor_sample
        / OUTPUT_SR
    )

    return anchor_sec, float(rms)

anchor位置の設定

In [23]:
def deterministic_uniform(
    key,
    low,
    high,
):
    """
    keyごとに再現可能な乱数を生成。
    """

    text = (
        f"{RANDOM_SEED}:{key}"
    )

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).digest()

    seed = int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    )

    rng = np.random.default_rng(seed)

    return float(
        rng.uniform(low, high)
    )

In [24]:
def choose_clip_range(
    anchor_sec,
    source_duration_sec,
    key,
):
    """
    anchorをclip内0.4～0.6秒へ配置しつつ、
    sourceから1秒が完全に取れる位置を選ぶ。
    """

    # clip_start = source_anchor - clip_anchor

    # start >= 0
    # → clip_anchor <= source_anchor

    # end <= duration
    # → clip_anchor >= source_anchor + 1 - duration

    valid_low = max(
        ANCHOR_MIN_SEC,
        anchor_sec
        + 1.0
        - source_duration_sec,
    )

    valid_high = min(
        ANCHOR_MAX_SEC,
        anchor_sec,
    )

    if valid_low > valid_high:
        return None

    if np.isclose(
        valid_low,
        valid_high,
    ):
        clip_anchor_sec = (
            valid_low
        )

    else:
        clip_anchor_sec = (
            deterministic_uniform(
                key,
                valid_low,
                valid_high,
            )
        )

    source_start_sec = (
        anchor_sec
        - clip_anchor_sec
    )

    start_sample = int(
        round(
            source_start_sec
            * OUTPUT_SR
        )
    )

    end_sample = (
        start_sample
        + OUTPUT_SAMPLES
    )

    return {
        "clip_anchor_sec":
            clip_anchor_sec,

        "source_start_sec":
            start_sample
            / OUTPUT_SR,

        "source_end_sec":
            end_sample
            / OUTPUT_SR,

        "start_sample":
            start_sample,

        "end_sample":
            end_sample,
    }

clipしたファイル名の生成

In [25]:
def make_safe_name(text):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(text),
    )

#### clippingの実行処理

In [26]:
def generate_clip(
    *,
    input_path,
    clip_id,
    source_id,
    source_dataset,
    source_recording_id,
    subject_id,
    provenance_group_id,
    class_id,
    class_name,
    usage_group,
    quality,
    region_start_sec=None,
    region_end_sec=None,
    event_id=None,
):
    """
    1 candidateから1秒clipを生成。
    """

    try:
        x = load_and_resample_wav(
            input_path
        )

    except Exception as e:
        return {
            "status": "decode_error",
            "drop_reason": repr(e),
            "clip_id": clip_id,
        }

    source_duration_sec = (
        len(x)
        / OUTPUT_SR
    )

    if len(x) < OUTPUT_SAMPLES:
        return {
            "status": "dropped",
            "drop_reason":
                "source_shorter_than_1s",
            "clip_id": clip_id,
        }

    # RMS最大位置
    anchor_sec, max_rms = (
        find_max_rms_anchor(
            x,
            region_start_sec,
            region_end_sec,
        )
    )

    if anchor_sec is None:
        return {
            "status": "dropped",
            "drop_reason":
                "rms_anchor_not_found",
            "clip_id": clip_id,
        }

    clip_range = choose_clip_range(
        anchor_sec,
        source_duration_sec,
        clip_id,
    )

    if clip_range is None:
        return {
            "status": "dropped",
            "drop_reason":
                "cannot_fit_1s_clip",
            "clip_id": clip_id,
        }

    start = clip_range[
        "start_sample"
    ]

    end = clip_range[
        "end_sample"
    ]

    clip = x[start:end]

    if len(clip) != OUTPUT_SAMPLES:
        return {
            "status": "dropped",
            "drop_reason":
                "unexpected_sample_count",
            "clip_id": clip_id,
        }

    safe_clip_id = make_safe_name(
        clip_id
    )

    output_path = (
        OUTPUT_ROOT
        / class_name
        / f"{safe_clip_id}.wav"
    )

    wavfile.write(
        output_path,
        OUTPUT_SR,
        clip,
    )

    return {
        "status":
            "generated",

        "drop_reason":
            "",

        "clip_id":
            clip_id,

        "source_id":
            source_id,

        "source_dataset":
            source_dataset,

        "source_recording_id":
            source_recording_id,

        "subject_id":
            subject_id,

        "provenance_group_id":
            provenance_group_id,

        "class_id":
            class_id,

        "class_name":
            class_name,

        "usage_group":
            usage_group,

        "quality":
            quality,

        "event_id":
            event_id,

        "source_event_start_sec":
            region_start_sec,

        "source_event_end_sec":
            region_end_sec,

        "source_anchor_sec":
            anchor_sec,

        "clip_anchor_sec":
            clip_range[
                "clip_anchor_sec"
            ],

        "source_start_sec":
            clip_range[
                "source_start_sec"
            ],

        "source_end_sec":
            clip_range[
                "source_end_sec"
            ],

        "anchor_rms":
            max_rms,

        "output_sr":
            OUTPUT_SR,

        "output_channels":
            1,

        "num_samples":
            len(clip),

        "relative_path":
            output_path.relative_to(
                OUTPUT_ROOT
            ).as_posix(),

        "split":
            "UNASSIGNED",
    }

#### ESC-50の処理

In [27]:
def process_esc50():

    manifest = pd.read_csv(
        ESC_MANIFEST
    )

    selected = manifest[
        manifest["usage_group"].isin(
            [
                "core",
                "extended",
                "target",
            ]
        )
    ].copy()

    results = []

    print(
        "ESC-50 candidates:",
        len(selected)
    )

    for i, row in selected.iterrows():

        input_path = (
            ESC_RAW
            / row["relative_path"]
        )

        clip_id = (
            "esc50__"
            + Path(
                row["source_recording_id"]
            ).stem
        )

        result = generate_clip(
            input_path=input_path,

            clip_id=clip_id,

            source_id=row["source_id"],

            source_dataset="ESC-50",

            source_recording_id=
                row["source_recording_id"],

            subject_id=pd.NA,

            provenance_group_id=
                row["provenance_group_id"],

            class_id=int(
                row["class_id"]
            ),

            class_name=
                row["class_name"],

            usage_group=
                row["usage_group"],

            quality=pd.NA,
        )

        results.append(result)
        if len(results) % 100 == 0:
            print(f"Processed: {len(results)}")

    return results

#### Coswaraの処理

In [28]:
def process_coswara():

    manifest = pd.read_csv(
        COSWARA_MANIFEST
    )

    selected = manifest[
        manifest["quality"].isin(
            [
                "excellent",
                "good",
            ]
        )
    ].copy()

    results = []

    print(
        "Coswara candidates:",
        len(selected)
    )

    for _, row in selected.iterrows():

        input_path = (
            COSWARA_RAW
            / row["relative_path"]
        )

        clip_id = (
            "coswara__"
            + make_safe_name(
                row[
                    "source_recording_id"
                ]
            )
        )

        result = generate_clip(
            input_path=input_path,

            clip_id=clip_id,

            source_id=row["source_id"],

            source_dataset="Coswara",

            source_recording_id=
                row["source_recording_id"],

            subject_id=
                row["subject_id"],

            provenance_group_id=
                row["provenance_group_id"],

            class_id=2,

            class_name="COUGH",

            usage_group="target",

            quality=row["quality"],
        )

        results.append(result)
        if len(results) % 100 == 0:
            print(f"Processed: {len(results)}")

    return results

#### AudioSetの処理

In [29]:
def process_audioset():

    sources = pd.read_csv(
        AUDIOSET_MANIFEST
    )

    events = pd.read_csv(
        AUDIOSET_EVENTS
    )

    # usable sourceだけ
    sources = sources[
        sources["availability"]
        == "available"
    ].copy()

    source_cols = sources[
        [
            "source_id",
            "source_recording_id",
            "relative_path",
            "provenance_group_id",
        ]
    ].copy()

    event_data = events.merge(
        source_cols,
        on=[
            "source_id",
            "provenance_group_id",
        ],
        how="inner",
        validate="many_to_one",
    )

    print(
        "AudioSet event candidates:",
        len(event_data)
    )

    results = []

    for _, row in event_data.iterrows():

        input_path = (
            AUDIOSET_RAW
            / row["relative_path"]
        )

        clip_id = (
            "audioset__"
            + make_safe_name(
                row["event_id"]
            )
        )

        result = generate_clip(
            input_path=input_path,

            clip_id=clip_id,

            source_id=
                row["source_id"],

            source_dataset=
                "AudioSet Strong",

            source_recording_id=
                row[
                    "source_recording_id"
                ],

            subject_id=pd.NA,

            provenance_group_id=
                row[
                    "provenance_group_id"
                ],

            class_id=1,

            class_name="SNORE",

            usage_group="target",

            quality=pd.NA,

            region_start_sec=float(
                row["event_start"]
            ),

            region_end_sec=float(
                row["event_end"]
            ),

            event_id=
                row["event_id"],
        )

        results.append(result)
        if len(results) % 100 == 0:
            print(f"Processed: {len(results)}")

    return results

#### clipされたデータセットの生成

In [30]:
all_results = []

print("=" * 60)
print("ESC-50")
print("=" * 60)

all_results.extend(
    process_esc50()
)

print()
print("=" * 60)
print("Coswara")
print("=" * 60)

all_results.extend(
    process_coswara()
)

print()
print("=" * 60)
print("AudioSet Strong")
print("=" * 60)

all_results.extend(
    process_audioset()
)

generation_log = pd.DataFrame(
    all_results
)

generation_log.to_csv(
    GENERATION_LOG,
    index=False,
)

print()
print("Generation log saved:")
print(GENERATION_LOG)

ESC-50
ESC-50 candidates: 1600
Processed: 100
Processed: 200
Processed: 300
Processed: 400
Processed: 500
Processed: 600
Processed: 700
Processed: 800
Processed: 900
Processed: 1000
Processed: 1100
Processed: 1200
Processed: 1300
Processed: 1400
Processed: 1500
Processed: 1600

Coswara
Coswara candidates: 5100
Processed: 100
Processed: 200
Processed: 300
Processed: 400
Processed: 500
Processed: 600
Processed: 700
Processed: 800
Processed: 900
Processed: 1000
Processed: 1100
Processed: 1200
Processed: 1300
Processed: 1400
Processed: 1500
Processed: 1600
Processed: 1700
Processed: 1800
Processed: 1900
Processed: 2000
Processed: 2100
Processed: 2200
Processed: 2300
Processed: 2400
Processed: 2500
Processed: 2600
Processed: 2700
Processed: 2800
Processed: 2900
Processed: 3000
Processed: 3100
Processed: 3200
Processed: 3300
Processed: 3400
Processed: 3500
Processed: 3600
Processed: 3700
Processed: 3800
Processed: 3900
Processed: 4000
Processed: 4100
Processed: 4200
Processed: 4300
Processed

#### manifestの作成

In [31]:
clip_manifest = (
    generation_log[
        generation_log["status"]
        == "generated"
    ]
    .copy()
    .reset_index(drop=True)
)

clip_manifest.to_csv(
    CLIP_MANIFEST,
    index=False,
)

print(
    "Generated clips:",
    len(clip_manifest)
)

print(
    "Clip manifest:",
    CLIP_MANIFEST
)

Generated clips: 7749
Clip manifest: /content/drive/MyDrive/TRON/AI/dataset/metadata/ai03/clip_manifest.csv


#### 検証

In [32]:
print("=== Generation status ===")

print(
    generation_log["status"]
    .value_counts(dropna=False)
)

print()
print("=== Drop reasons ===")

print(
    generation_log[
        generation_log["status"]
        != "generated"
    ]["drop_reason"]
    .value_counts(dropna=False)
)

print()
print("=== Dataset ===")

print(
    clip_manifest[
        "source_dataset"
    ].value_counts()
)

print()
print("=== Class ===")

print(
    clip_manifest[
        "class_name"
    ].value_counts()
)

print()
print("=== Usage group ===")

print(
    clip_manifest[
        "usage_group"
    ].value_counts()
)

print()
print("=== Quality ===")

print(
    clip_manifest[
        "quality"
    ].value_counts(
        dropna=False
    )
)

print()
print("=== Anchor ===")

print(
    clip_manifest[
        "clip_anchor_sec"
    ].describe()
)

print()
print("=== Samples ===")

print(
    clip_manifest[
        "num_samples"
    ].value_counts()
)


# ============================================================
# Assertions
# ============================================================

assert (
    clip_manifest["output_sr"]
    == 16000
).all()

assert (
    clip_manifest["output_channels"]
    == 1
).all()

assert (
    clip_manifest["num_samples"]
    == 16000
).all()

assert (
    clip_manifest["clip_anchor_sec"]
    >= 0.4
).all()

assert (
    clip_manifest["clip_anchor_sec"]
    <= 0.6
).all()

assert (
    clip_manifest["split"]
    == "UNASSIGNED"
).all()

assert (
    clip_manifest["clip_id"]
    .is_unique
)

print()
print("AI-03 manifest check: OK")

=== Generation status ===
status
generated    7749
dropped        60
Name: count, dtype: int64

=== Drop reasons ===
drop_reason
rms_anchor_not_found      42
cannot_fit_1s_clip        14
source_shorter_than_1s     4
Name: count, dtype: int64

=== Dataset ===
source_dataset
Coswara            5082
ESC-50             1600
AudioSet Strong    1067
Name: count, dtype: int64

=== Class ===
class_name
COUGH    5122
OTHER    1520
SNORE    1107
Name: count, dtype: int64

=== Usage group ===
usage_group
target      6229
extended     800
core         720
Name: count, dtype: int64

=== Quality ===
quality
excellent    4544
<NA>         2667
good          538
Name: count, dtype: int64

=== Anchor ===
count    7749.000000
mean        0.497848
std         0.059989
min         0.400000
25%         0.445457
50%         0.497195
75%         0.549692
max         0.600000
Name: clip_anchor_sec, dtype: float64

=== Samples ===
num_samples
16000.0    7749
Name: count, dtype: int64

AI-03 manifest check: OK


In [33]:
invalid_files = []

for _, row in clip_manifest.iterrows():

    path = (
        OUTPUT_ROOT
        / row["relative_path"]
    )

    try:

        with wave.open(
            str(path),
            "rb",
        ) as wf:

            sr = wf.getframerate()
            ch = wf.getnchannels()
            width = wf.getsampwidth()
            frames = wf.getnframes()

        if not (
            sr == 16000
            and ch == 1
            and width == 2
            and frames == 16000
        ):
            invalid_files.append(
                (
                    row["clip_id"],
                    sr,
                    ch,
                    width,
                    frames,
                )
            )

    except Exception as e:

        invalid_files.append(
            (
                row["clip_id"],
                repr(e),
            )
        )


print(
    "Invalid WAV files:",
    len(invalid_files)
)

if invalid_files:
    print(
        invalid_files[:20]
    )

assert len(invalid_files) == 0

print(
    "AI-03 WAV validation: OK"
)

Invalid WAV files: 0
AI-03 WAV validation: OK


In [34]:
dropped = generation_log[
    generation_log["status"] != "generated"
].copy()

print("=== Drop reason x Dataset ===")
print(
    pd.crosstab(
        dropped["drop_reason"],
        dropped.get("source_dataset"),
        margins=True,
    )
)

print()
print("=== Dropped rows ===")

cols = [
    c for c in [
        "clip_id",
        "source_dataset",
        "drop_reason",
        "event_id",
    ]
    if c in dropped.columns
]

print(
    dropped[cols]
    .to_string(index=False)
)

=== Drop reason x Dataset ===
Empty DataFrame
Columns: []
Index: []

=== Dropped rows ===
                                                     clip_id source_dataset            drop_reason event_id
  coswara__20200418_C7Km0KttQRMMM6UoyocajfgZAOB3_cough-heavy            NaN source_shorter_than_1s      NaN
coswara__20200418_C7Km0KttQRMMM6UoyocajfgZAOB3_cough-shallow            NaN source_shorter_than_1s      NaN
  coswara__20200814_kgjTguvo3vZJTO7F1qO9GxEicbA3_cough-heavy            NaN source_shorter_than_1s      NaN
coswara__20200919_hVZQcrz6cfaOMfL3jpRqJBUiumF2_cough-shallow            NaN     cannot_fit_1s_clip      NaN
coswara__20200930_MF5f99sONpcYl5fYsVzh32XwSmm2_cough-shallow            NaN     cannot_fit_1s_clip      NaN
coswara__20201031_E8foKQD2BqYfZjFhdhknvaDp3Ot2_cough-shallow            NaN     cannot_fit_1s_clip      NaN
coswara__20210206_HDZ7Et3VldU238svx2iusUhQ16p2_cough-shallow            NaN     cannot_fit_1s_clip      NaN
coswara__20210406_qSsnWdo1MtOiGJJmH3oOXcz90Ry1